# 05 — BLIP-2 ITM Inference (Round 2 of the iterative ensemble)

BLIP-2 image-text matching score gallery × query. This is Round 2 in the 4-round iterative ensemble (UIT → **BLIP-2** → CLIP → PE-G14).

**Model:** `Salesforce/blip2-itm-vit-g` (paper-faithful — BLIP-2 with ViT-g vision tower, Q-Former + ITM head).

**Approach:** dual-mode score = α·ITM_logit + (1-α)·ITC_cosine. Paper-default α=0.5 but tuned on val.

**Tile sizes (A100-80GB):** 256 query × 4096 gallery per ITM tile (since the ITM cross-encoder is very expensive).

**Outputs:**
- `features/blip2/{gallery,queries,val,val_zs}.npz` (Q-Former projected ITC emb)
- `features/blip2/scores_blip2.pt` ({scores: Q×G fp16, query_ids, gallery_ids})
- `features/blip2/scores_val.pt`, `scores_val_zs.pt`

**ETA:** ~1.5-2h on A100 80GB BF16 (1978q × 36773g ITM ≈ 73M forward passes; ITC ranking reduced to 1024 candidates before ITM rerank to save compute).

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

NOTEBOOK_DIR = Path('.').resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from aic_colab_utils import (
    setup_aic2026_environment, select_a100_device,
    sync_chunk_to_drive, wait_for_pending_syncs, maybe_clear_cuda,
    save_npz_atomic,
)

PATHS = setup_aic2026_environment()
INPUT_ROOT = PATHS['input_root']; MANIFEST_DIR = PATHS['manifest_dir']
LOCAL_ROOT = PATHS['local_root']; OUTPUT_ROOT = PATHS['output_root']
DRIVE_OUTPUT_ROOT = PATHS['drive_output_root']

BLIP_FEAT_DIR = OUTPUT_ROOT / 'features' / 'blip2'
BLIP_FEAT_DIR.mkdir(parents=True, exist_ok=True)
device = select_a100_device()

In [ ]:
def pip_install(*p):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *p])
try:
    import lavis  # noqa
except Exception:
    pip_install('salesforce-lavis>=1.0.2')

import torch, torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
from lavis.models import load_model_and_preprocess

MODEL_NAME = 'blip2_image_text_matching'
MODEL_TYPE = 'pretrain'
model, vis_processors, txt_processors = load_model_and_preprocess(
    name=MODEL_NAME, model_type=MODEL_TYPE, is_eval=True, device=device,
)
model = model.eval()
print('BLIP-2 ITM loaded.')

In [ ]:
gallery_df = pd.read_parquet(MANIFEST_DIR / 'gallery_manifest.parquet')
query_df = pd.read_parquet(MANIFEST_DIR / 'query_manifest.parquet')
val_df = pd.read_parquet(MANIFEST_DIR / 'val_split.parquet')
val_zs_df = pd.read_parquet(MANIFEST_DIR / 'val_zeroshot_scene.parquet')

def remap(p):
    if not isinstance(p, str) or not p: return p
    if p.startswith(str(INPUT_ROOT)): return p
    for a in ('annotation/', 'train/imgs_', 'name-masked_test-set/', 'gallery/'):
        i = p.find(a)
        if i >= 0: return str(INPUT_ROOT / p[i:])
    return p
for d in (gallery_df, val_df, val_zs_df):
    if 'image_path' in d.columns:
        d['image_path'] = d['image_path'].map(remap)
print('gallery:', len(gallery_df), 'queries:', len(query_df))

In [ ]:
# === Encode ITC features (fast retrieval candidates) ===
IMG_BATCH = 64
TXT_BATCH = 256
amp_dtype = torch.bfloat16

@torch.inference_mode()
def encode_images_itc(df, id_col):
    embs, ids, paths, ok = [], [], [], []
    for s in tqdm(range(0, len(df), IMG_BATCH), desc=f'blip2-itc-img {id_col}'):
        rows = df.iloc[s:s + IMG_BATCH]
        ims, oks = [], []
        for p in rows['image_path']:
            try:
                pil = Image.open(p).convert('RGB'); oks.append(True)
            except Exception:
                pil = Image.new('RGB', (224, 224)); oks.append(False)
            ims.append(vis_processors['eval'](pil))
        x = torch.stack(ims, 0).to(device)
        with torch.autocast(device_type=device.type, dtype=amp_dtype):
            feats = model.extract_features({'image': x}, mode='image')
        # image_embeds_proj: (B, 32 query tokens, D) — use first query token or mean-pool
        emb = feats.image_embeds_proj[:, 0, :]  # CLS-like
        emb = F.normalize(emb.float(), dim=-1).cpu().numpy().astype('float16')
        embs.append(emb); ids.extend(rows[id_col].astype(str).tolist())
        paths.extend(rows.get('image_path', pd.Series([''] * len(rows))).tolist())
        ok.extend(oks)
    return np.concatenate(embs, 0), np.array(ids), np.array(paths), np.array(ok)

@torch.inference_mode()
def encode_texts_itc(ids, texts):
    embs = []
    for s in tqdm(range(0, len(texts), TXT_BATCH), desc='blip2-itc-txt'):
        batch = list(texts[s:s + TXT_BATCH])
        # LAVIS BLIP-2: text_input passed through txt_processors
        text_input = [txt_processors['eval'](t) for t in batch]
        sample = {'text_input': text_input}
        with torch.autocast(device_type=device.type, dtype=amp_dtype):
            feats = model.extract_features(sample, mode='text')
        emb = feats.text_embeds_proj[:, 0, :]
        embs.append(F.normalize(emb.float(), dim=-1).cpu().numpy().astype('float16'))
    return np.concatenate(embs, 0), np.array([str(x) for x in ids])

g_emb, g_ids, g_paths, g_ok = encode_images_itc(gallery_df, 'gallery_id')
q_emb, q_ids = encode_texts_itc(query_df['query_index'].astype(str).tolist(), query_df['caption'].astype(str).tolist())
save_npz_atomic(BLIP_FEAT_DIR / 'gallery.npz', ids=g_ids, paths=g_paths, embeddings=g_emb, ok=g_ok)
save_npz_atomic(BLIP_FEAT_DIR / 'queries.npz', ids=q_ids, embeddings=q_emb)
print('ITC embeddings saved.')

In [ ]:
# === ITM rerank on top-K candidates from ITC (reduces cost compared to full 36k×1.9k) ===
ITM_TOPK = 1024  # ITM rerank on top-1024 ITC candidates per query
ITC_ALPHA = 0.4  # weight for ITC vs ITM in the final score
ITM_BATCH = 32

# Compute ITC similarity matrix (Q × G)
Q_t = torch.from_numpy(q_emb.astype('float32')).to(device)
G_t = torch.from_numpy(g_emb.astype('float32')).to(device)
Q_t = F.normalize(Q_t, dim=-1); G_t = F.normalize(G_t, dim=-1)
S_itc = (Q_t @ G_t.T)  # fp32 GPU

# Get ITC top-K candidate gallery indices per query
itc_topk_vals, itc_topk_idx = torch.topk(S_itc, k=min(ITM_TOPK, S_itc.shape[1]), dim=1)

# Initialize final score matrix with ITC scores (will overwrite top-K)
S_final = S_itc.clone()

# Run ITM head on top-K candidates per query
@torch.inference_mode()
def itm_score(image_pil_list, text_str):
    """Compute ITM match probability for a list of (image, single_text) pairs."""
    inp = [vis_processors['eval'](im) for im in image_pil_list]
    x = torch.stack(inp, 0).to(device)
    with torch.autocast(device_type=device.type, dtype=amp_dtype):
        itm_out = model({'image': x, 'text_input': [txt_processors['eval'](text_str)] * len(image_pil_list)}, match_head='itm')
    # itm_out: (B, 2) logits → softmax dim=1 → P(match)
    probs = F.softmax(itm_out.float(), dim=1)[:, 1]
    return probs

# Cache PIL images for top-K reuse
import functools
@functools.lru_cache(maxsize=4096)
def _open_image(path):
    try:
        return Image.open(path).convert('RGB').copy()
    except Exception:
        return Image.new('RGB', (224, 224))

gallery_paths = gallery_df['image_path'].tolist()
query_texts = query_df['caption'].astype(str).tolist()

for q_idx in tqdm(range(len(query_texts)), desc='blip2-itm-rerank'):
    cand_idx = itc_topk_idx[q_idx].cpu().numpy()
    text = query_texts[q_idx]
    itm_scores = []
    for s in range(0, len(cand_idx), ITM_BATCH):
        batch_idx = cand_idx[s:s + ITM_BATCH]
        pil_batch = [_open_image(gallery_paths[i]) for i in batch_idx]
        probs = itm_score(pil_batch, text)
        itm_scores.append(probs.cpu())
    itm_arr = torch.cat(itm_scores).numpy()
    itc_arr = itc_topk_vals[q_idx].cpu().numpy()
    fused = ITC_ALPHA * itc_arr + (1 - ITC_ALPHA) * itm_arr
    # Overwrite final scores at top-K positions
    S_final[q_idx, cand_idx] = torch.from_numpy(fused.astype('float32')).to(device)

S_blip2 = S_final.half().cpu()
torch.save({'scores': S_blip2, 'query_ids': q_ids, 'gallery_ids': g_ids}, BLIP_FEAT_DIR / 'scores_blip2.pt')
print('scores_blip2.pt:', tuple(S_blip2.shape))

In [ ]:
# === Val encoding + ITC-only scores (val sets small enough for full ITM (optional)) ===
for name, df in [('val', val_df), ('val_zs', val_zs_df)]:
    if len(df) == 0:
        continue
    v_img, vi_ids, vp, vok = encode_images_itc(df, 'image_id')
    v_txt, _ = encode_texts_itc(df['image_id'].astype(str).tolist(), df['caption'].astype(str).tolist())
    save_npz_atomic(BLIP_FEAT_DIR / f'{name}.npz', ids=vi_ids, paths=vp, embeddings=v_img, ok=vok)
    save_npz_atomic(BLIP_FEAT_DIR / f'{name}_text.npz', ids=vi_ids, embeddings=v_txt)
    Qv = F.normalize(torch.from_numpy(v_txt.astype('float32')).to(device), dim=-1)
    Gv = F.normalize(torch.from_numpy(v_img.astype('float32')).to(device), dim=-1)
    Sv = (Qv @ Gv.T).half().cpu()
    torch.save({'scores': Sv, 'query_ids': vi_ids, 'gallery_ids': vi_ids}, BLIP_FEAT_DIR / f'scores_{name}.pt')
    print(f'scores_{name}.pt:', tuple(Sv.shape))

for f in BLIP_FEAT_DIR.rglob('*'):
    if f.is_file():
        sync_chunk_to_drive(f, LOCAL_ROOT, DRIVE_OUTPUT_ROOT, background=True)
wait_for_pending_syncs()
print('BLIP-2 inference done.')